# Phase B0 — one-pass cloud-native GPU development

This is a new Colab-native experiment, not a recovery of the aborted Mac attempt. One fresh Tesla T4 runtime generates the frozen Linux data, trains both Phase B0 variants on CUDA, evaluates them, and writes persistent artifacts to Drive. There is no cross-platform hash exchange.


## 1. Confirm a fresh Tesla T4 runtime


In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then reconnect.')
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name)
if gpu_name != 'Tesla T4':
    raise RuntimeError(f'This frozen run requires Tesla T4, found {gpu_name!r}.')


## 2. Upload the single authorized execution bundle


In [ ]:
from google.colab import files
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.tar.gz')]
if len(archives) != 1:
    raise RuntimeError('Upload exactly one cloud-native .tar.gz bundle.')
ARCHIVE = archives[0]
print('Uploaded:', ARCHIVE)


## 3. Restore source and the six frozen comparator resources


In [ ]:
import json, os, pathlib, shutil, subprocess, sys, tarfile
extract_root = pathlib.Path('/content/phase_b0_cloud_native_bundle')
repo_root = pathlib.Path('/content/latent-stroke-dynamics')
if extract_root.exists() or repo_root.exists():
    raise RuntimeError('Bundle extraction already exists; use a fresh runtime.')
extract_root.mkdir()
with tarfile.open(ARCHIVE, 'r:gz') as archive:
    archive.extractall(extract_root, filter='data')
manifest = json.loads((extract_root / 'bundle_manifest.json').read_text())
if manifest['status'] != 'phase_b0_colab_native_execution_bundle_authorized_once':
    raise RuntimeError('Unexpected execution bundle status.')
if manifest['new_experiment'] is not True or manifest['recovery_of_mac_attempt'] is not False:
    raise RuntimeError('Bundle is not the frozen cloud-native experiment.')
if manifest['cloud_native_development_authorized'] is not True:
    raise RuntimeError('Bundle is not authorized for training.')
if manifest['formal_authorized'] is not False or manifest['phase_b1_authorized'] is not False or manifest['phase_b2_authorized'] is not False:
    raise RuntimeError('A later phase was unexpectedly authorized.')
subprocess.run(['git', 'clone', '--branch', manifest['branch'], str(extract_root / 'repository.bundle'), str(repo_root)], check=True)
head = subprocess.check_output(['git', '-C', str(repo_root), 'rev-parse', 'HEAD'], text=True).strip()
if head != manifest['source_commit']:
    raise RuntimeError('Restored Git commit differs from the bundle manifest.')
for source in (extract_root / 'resources').rglob('*'):
    if source.is_file():
        destination = repo_root / source.relative_to(extract_root / 'resources')
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
print('Restored commit:', head)
print('Frozen resources:', manifest['resource_count'])


## 4. Install once and run the complete gate


In [ ]:
%cd /content/latent-stroke-dynamics
%pip install -q -e ".[dev]"
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


Expected: **160 passed**. Any failure stops before Drive mounting or scientific execution.


## 5. Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
artifact_root = pathlib.Path('/content/drive/MyDrive/latent-stroke-dynamics-phase-b0-cloud-native')
if str(artifact_root) != manifest['artifact_root']:
    raise RuntimeError('Drive artifact root differs from the authorization.')
final_root = artifact_root / 'phase-b0-cloud-native-development-2026-08-24'
incomplete_root = artifact_root / 'phase-b0-cloud-native-development-2026-08-24.incomplete'
log_path = artifact_root / 'phase-b0-cloud-native-console.log'
if final_root.exists() or incomplete_root.exists() or log_path.exists():
    raise RuntimeError('This one-time experiment already started or completed. Do not run again.')


## 6. One explicit training switch
Change the single value below only after Steps 1–5 pass. The next cell generates the Linux data and then trains both new models on the GPU in this same runtime.


In [ ]:
RUN_CLOUD_NATIVE_TRAINING = False
if RUN_CLOUD_NATIVE_TRAINING is not True:
    raise RuntimeError('Paused. Change RUN_CLOUD_NATIVE_TRAINING to True for the authorized run.')


## 7. Generate data, train on CUDA, evaluate, and save to Drive


In [ ]:
artifact_root.mkdir(parents=True, exist_ok=True)
command = [sys.executable, 'experiments/28_phase_b_colab_native_development.py', '--development', '--artifact-root', str(artifact_root)]
environment = dict(os.environ)
environment['PYTHONUNBUFFERED'] = '1'
with log_path.open('x', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=environment)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'Cloud-native training exited with code {return_code}. Preserve Drive artifacts and do not rerun.')


## 8. Download one small completion handoff


In [ ]:
if not final_root.is_dir() or incomplete_root.exists():
    raise RuntimeError('Execution did not finalize; preserve Drive artifacts and do not rerun.')
decision = json.loads((final_root / 'decision.json').read_text())
run_config = json.loads((final_root / 'run_config.json').read_text())
integrity = json.loads((final_root / 'integrity_manifest.json').read_text())
handoff = {
    'status': 'phase_b0_colab_native_development_complete',
    'source_commit': head,
    'final_root': str(final_root),
    'decision': decision,
    'run_config': run_config,
    'integrity_manifest': integrity,
    'console_log': str(log_path),
    'new_experiment': True,
    'recovery_of_mac_attempt': False,
    'do_not_rerun': True
}
handoff_path = pathlib.Path('/content/phase-b0-cloud-native-completion-handoff.json')
handoff_path.write_text(json.dumps(handoff, indent=2) + '\n')
print(json.dumps(handoff, indent=2, sort_keys=True))
files.download(str(handoff_path))
